# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhanish0711/FlyRank-Machine-Learning-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This notebook maps **Lane 2 (Refresh / Content Opportunity Scoring)** onto the machine learning system loop: defining the task type, target label origin, evaluation metric, unit of analysis dataframe, and empirical justification for ML over hand rules.

## 1. My lane as an ML task (type)

**Chosen Lane:** **Lane 2 — Refresh / Content Opportunity Scoring**

**Task Type:** **Ranking / Pointwise Priority Scoring**

**Why Ranking / Scoring instead of pure Binary Classification?**
In content operations, editorial bandwidth is strictly capacity-constrained. An editorial team cannot review 1,000 positive classification flags simultaneously; they need an ordered queue so they can review the top 20 or top 50 highest-impact pages each week. We frame the task as **Pointwise Ranking**: training a model to output a calibrated probability score $P(\text{decline} \mid X)$, and sorting pages in descending order of risk weighted by traffic demand. This directly supports the editor's workflow by surfacing the highest-value, highest-risk content candidates at the top of the queue.

In [1]:
# Declaration of ML Task Type
task_meta = {
    'Lane': 'Lane 2 - Refresh / Content Opportunity Scoring',
    'ML Task Type': 'Ranking / Pointwise Scoring',
    'Primary Output': 'Ranked Opportunity Queue (Top-K items)',
    'Action Supported': 'Weekly Editorial Review Allocation'
}
for k, v in task_meta.items():
    print(f'{k:18s}: {v}')


Lane              : Lane 2 - Refresh / Content Opportunity Scoring
ML Task Type      : Ranking / Pointwise Scoring
Primary Output    : Ranked Opportunity Queue (Top-K items)
Action Supported  : Weekly Editorial Review Allocation


## 2. Target or proxy

### Target Definition
* **Starter Dataset Proxy Label:** `is_declining_label = (df['trend_direction'].str.lower() == 'down').astype(int)`
* **Full Warehouse Target:** Future organic traffic decline measured over a subsequent 30-day window (`clicks_next30` / `clicks_prev30` < 0.85) relative to a preceding 90-day feature window.

### Where does this label come from?
The target is derived from an **OBSERVED OUTCOME** measured in real search traffic data. It is **not** defined by a hand-written product rule or internal priority flag (such as `health_score` or `needs_refresh_flag`). Using an observed traffic outcome ensures the model learns real empirical patterns from search behavior rather than merely reproducing a human rule (avoiding circular logic).

In [2]:
import pandas as pd
from pathlib import Path

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)

label_counts = df['is_declining_label'].value_counts()
label_pct = df['is_declining_label'].value_counts(normalize=True) * 100

print('=== Target Label Distribution (is_declining_label) ===')
print(f'Stable / Growing (0): {label_counts[0]:,} rows ({label_pct[0]:.1f}%)')
print(f'Declining (1)       : {label_counts[1]:,} rows ({label_pct[1]:.1f}%)')
print(f'Base Rate           : {df["is_declining_label"].mean():.3f}')


=== Target Label Distribution (is_declining_label) ===
Stable / Growing (0): 13,738 rows (45.8%)
Declining (1)       : 16,262 rows (54.2%)
Base Rate           : 0.542


## 3. Success metric

### Primary Metric: Precision@50 (and Precision@20)
Because editors process review queues in fixed weekly batches (typically 20 to 50 articles), generic classification accuracy or ROC-AUC is misleading. The metric that directly reflects operational success is **Precision@K**:

$$\text{Precision@K} = \frac{\text{True Declining Pages in Top } K}{K}$$

### Why Precision@50?
If an editor reviews 50 pages from our queue, Precision@50 measures what percentage of those 50 pages were actually declining. A Precision@50 of **0.740** means 37 of the 50 reviewed pages were true opportunities, minimising wasted editorial budget.

### Secondary Metric: Average Precision (AP) / PR-AUC
Average Precision measures ranking quality across all threshold cuts, rewarding models that push true declining pages to the very top of the list.

In [3]:
import numpy as np

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk_labels = np.asarray(labels)[order[:k]]
    return topk_labels.mean()

# Random dummy baseline for demonstration
dummy_scores = np.random.rand(len(df))
p20_random = precision_at_k(dummy_scores, df['is_declining_label'], 20)
p50_random = precision_at_k(dummy_scores, df['is_declining_label'], 50)

print('=== Success Metric Baseline Benchmark ===')
print(f'Random Ranking Precision@20: {p20_random:.3f}')
print(f'Random Ranking Precision@50: {p50_random:.3f}')
print(f'Dataset Base Rate          : {df["is_declining_label"].mean():.3f}')


=== Success Metric Baseline Benchmark ===
Random Ranking Precision@20: 0.600
Random Ranking Precision@50: 0.440
Dataset Base Rate          : 0.542


## 4. The unit of analysis, as a real dataframe

**Unit of Analysis Grain:** **One row = One pseudonymized content item (`content_id`) for a specific client (`client_id`) over a 90-day observation window.**

Below we inspect the slice of data representing our unit of analysis:

In [4]:
print(f'Dataframe Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Unique content_ids: {df["content_id"].nunique():,}')
print(f'Unique client_ids : {df["client_id"].nunique():,}')

key_cols = [
    'content_id', 'client_id', 'impressions_90d', 'sessions_90d', 
    'days_since_last_update', 'avg_position', 'ctr', 'is_declining_label'
]
df[key_cols].head(5)


Dataframe Shape: 30,000 rows x 45 columns
Unique content_ids: 30,000
Unique client_ids : 32


,content_id,client_id,impressions_90d,sessions_90d,days_since_last_update,avg_position,ctr,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,17,20,10.6,0.76,1
1,content_a1fb4e703a9e,client_4e07408562,15320,9,25,20.3,0.05,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,20,36.5,0.09,1
3,content_331d6c4de07b,client_19581e27de,11751,78,22,6.2,0.49,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,14,44.0,0.13,1


## 5. Why ML beats a fixed rule here

### Why static rules fail
A transparent hand rule (such as `stale_visible = (days_since_last_update >= 180) & (impressions_90d >= 500)`) relies on fixed linear cutoffs. However, real search decay involves non-linear interactions:
1. A page with 10,000 impressions updated 90 days ago might be decaying faster than a page with 500 impressions updated 200 days ago.
2. CTR expectations vary non-linearly by position tier (Position 1 vs Position 8).
3. Content age, intent type, and engagement rate combine in tangled ways that rigid if-statements cannot capture.

### Empirical Proof: Hand Rule vs Learned Model
Below we compute and compare the actual Precision@50 of a hand-written rule versus a learned model on the dataset:

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# 1. Hand Rule Score: Stale x Visible x Impressions
stale = (df['days_since_last_update'] >= 180).astype(int)
visible = (df['impressions_90d'] >= 500).astype(int)
df['hand_rule_score'] = stale * visible * df['impressions_90d']

# 2. Learned Model (Random Forest on observable features)
features = [
    'content_age_days', 'days_since_last_update', 'impressions_90d', 
    'sessions_90d', 'avg_position', 'ctr', 'word_count', 'engagement_rate'
]
X = df[features].fillna(0)
y = df['is_declining_label'].values

X_train, X_test, y_train, y_test, df_tr, df_te = train_test_split(
    X, y, df, test_size=0.3, random_state=42, stratify=df['client_id']
)

rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
model_scores = rf.predict_proba(X_test)[:, 1]
hand_scores = df_te['hand_rule_score'].values

p50_hand = precision_at_k(hand_scores, y_test, 50)
p50_model = precision_at_k(model_scores, y_test, 50)

print('=== Empirical Comparison: Hand Rule vs Learned ML Model ===')
print(f'Hand-Written Rule  Precision@50: {p50_hand:.3f}  (~{int(p50_hand*50)} / 50 correct)')
print(f'Learned RF Model   Precision@50: {p50_model:.3f}  (~{int(p50_model*50)} / 50 correct)')
print(f'Precision Gain                 : {p50_model / (p50_hand + 1e-6):.2f}x improvement')


=== Empirical Comparison: Hand Rule vs Learned ML Model ===
Hand-Written Rule  Precision@50: 0.660  (~33 / 50 correct)
Learned RF Model   Precision@50: 0.860  (~43 / 50 correct)
Precision Gain                 : 1.30x improvement


## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.